## Business Understanding

### Problem Statements
Berdasarkan kondisi yang telah dijelaskan sebelumnya, muncul beberapa pertanyaan penting, yaitu:
- Bagaimana sistem rekomendasi berbasis kesamaan kategori produk dapat membantu pengguna menemukan produk lain yang relevan?
- Bagaimana sistem ini dapat memberikan rekomendasi dengan memanfaatkan informasi kategori dan tingkat popularitas produk berdasarkan rating?
- Bagaimana sistem ini dapat merekomendasikan produk berdasarkan kategori serta kualitas produk yang diukur melalui jumlah rekomendasi dari pengguna?

### Goals
Proyek analisis prediktif ini bertujuan utama untuk menjawab pertanyaan-pertanyaan tersebut. Secara lebih rinci, tujuan yang ingin dicapai meliputi:
- Menyediakan rekomendasi produk yang relevan menggunakan pendekatan Content-Based Filtering berbasis kesamaan kategori produk
- Menggunakan informasi kategori dan rating produk untuk menampilkan produk-produk serupa yang lebih relevan bagi pengguna
- Menggunakan informasi kategori serta jumlah total rekomendasi dari pengguna untuk menampilkan produk serupa yang lebih akurat

### Solution statements
Solusi yang ditawarkan adalah dengan menerapkan Content-Based Filtering menggunakan metode TF-IDF dan Cosine Similarity.

Pendekatan ini menggunakan representasi teks produk dengan teknik TF-IDF (Term Frequency-Inverse Document Frequency) untuk mengukur tingkat pentingnya kata-kata pada kategori produk. Setiap produk akan diubah menjadi vektor berdasarkan kata-kata kunci dalam kategorinya, lalu dihitung kesamaan antar produk menggunakan cosine similarity. Dengan metode ini, sistem dapat merekomendasikan produk kepada pengguna berdasarkan kesamaan kategori dengan produk yang sebelumnya mereka pilih.


## Data Understanding

Dataset berisi 3961 baris dan 6 kolom, dataset terdiri dari Kolom-kolomnya berisi brand_name, product_name, product_id, beauty_point_earned, price_range, price_by_combinations, url, active_date, default_category, categories, rating_types_str, average_rating, total_reviews, average_rating_by_types, total_recommended_count, total_repurchase_maybe_count, total_repurchase_no_count, total_repurchase_yes_count, total_in_wishlist. Variabel yang akan digunakan pada kasus kali ini sebagai parameter rekomendasi adalah variabel default_category. Kondisi data masih belum bersih dengan ditandai masih adanya missing values.

Referensi:
Nugroho. "Product Data E-Commerce Lazada". Tautan: [https://www.kaggle.com/datasets/intodarkmoon/product-data-e-commerce-lazada]. Diakses pada 27 April 2025.

## Data Loading

In [87]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
import seaborn as sns

In [88]:
products = pd.read_csv('lazada_data.csv')
products

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
0,Official Redmi A3 | Layar Muluz 90 Hz berukura...,4.936567,1072.0,Xiaomi,Kota Depok,1199000.0
1,Infinix Note 40 8/256GB - Up to 16GB Extended ...,4.938308,1702.0,Infinix,Kota Depok,2729000.0
2,Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Ex...,4.931507,73.0,Infinix,Kota Depok,3579000.0
3,[PROMO] VIVO Y15S RAM 3GB / ROM 64GB - FULLSET...,5.000000,23.0,Mentari Cell,Kab. Madiun,999000.0
4,Samsung Galaxy A05 4/64 4/128,NaN,NaN,Ady Phone Store,Kab. Kebumen,1479000.0
...,...,...,...,...,...,...
3956,Samsung a12 Ram 6/128 mulus masih ori gransi r...,NaN,NaN,CANTIKA AMANAH CELULLER,Kab. Bogor,1725000.0
3957,SAMSUNG M10 2/16GB GARANSI RESMI SEIN ORIGINAL...,NaN,NaN,Honchi Celuller,Kota Tangerang,1575000.0
3958,Xiaomi redmi note 13 pro 5G 8/256 garansi resmi,NaN,NaN,Galeri-Hp 040595,Kota Jakarta Pusat,4398000.0
3959,samsung s21 ultra 256gb,NaN,NaN,Istoree,Kota Bandung,15650000.0


- Data berhasil terpanggil, di sini data terdiri dari 3961 baris dan 6 kolom.
- Kolom-kolomnya berisi Name, Rating Score, Total Review, Seller Name, Location, dan Price (IDR).

## Univariate Exploratory Data Analysis

Variabel-variabel pada dataset adalah sebagai berikut:
- Name: nama produk
- Rating Score: penilaian produk oleh customer
- Total Review: Berapa banyak customer yang melakukan review
- Seller Name: Nama penjual.
- Location: lokasi toko offline / gudang.
- Price (IDR): Harga barang dalam mata uang Rupiah.


In [89]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3961 entries, 0 to 3960
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          3961 non-null   object 
 1   Rating Score  1720 non-null   float64
 2   Total Review  1726 non-null   float64
 3   Seller Name   3961 non-null   object 
 4   Location      3961 non-null   object 
 5   Price (IDR)   3961 non-null   float64
dtypes: float64(3), object(3)
memory usage: 185.8+ KB


In [90]:
print('Jumlah Name: ', len(products.Name.unique()))
print('Jumlah Seller Name: ', len(products['Seller Name'].unique()))
print('Jumlah Rating Score: ', len(products['Rating Score'].unique()))
print('Jumlah Location: ', len(products.Location.unique()))

Jumlah Name:  3796
Jumlah Seller Name:  304
Jumlah Rating Score:  510
Jumlah Location:  69


Dengan fungsi unnique, dapat diketahui jika dataaset terdiri dari 3796 nama produk yang berbeda, 304 nama toko yang berbeda, 510 nilai rating yang berbeda, dan 69 lokasi pedagang yang berbeda

## Data Preparation

#### Memeriksa Data Terduplikasi

In [91]:
products.duplicated().sum()

np.int64(9)

Data memiliki 9 data terduplikasi. Selanjutnya akan menghilangkan data terduplikasi tersebut.

In [92]:
products.drop_duplicates(inplace=True)

In [93]:
products.duplicated().sum()

np.int64(0)

Sekarang, data duplikasi sudah tidak ada.

#### Mengatasi Missing Value

In [94]:
products.describe()

,Rating Score,Total Review,Price (IDR)
count,1720.000000,1726.000000,3.952000e+03
mean,4.840749,131.239282,3.373935e+06
std,0.435869,550.874492,3.637555e+06
min,1.000000,0.000000,1.000000e+05
25%,4.869672,2.000000,1.450000e+06
50%,5.000000,9.000000,2.295000e+06
75%,5.000000,48.000000,3.899000e+06
max,5.000000,11409.000000,5.204700e+07


In [95]:
products.isnull().sum()

,0
Name,0
Rating Score,2232
Total Review,2226
Seller Name,0
Location,0
Price (IDR),0


terdapat banyak missing values di dua kolom, yaitu 'Rating Score' dan 'Total Review'. Oleh sebab itu, baris pada dataset yang memiliki missing value pada kolom tersebut akan di drop, karena memiliki missing values dan nilai yang tidak terlalu berpengaruh terhadap tujuan rekomendasi.

In [96]:
#Baris yang tidak terlalu berpengaruh dan memiliki NaN didrop
products.dropna(subset=['Rating Score', 'Total Review'], inplace=True)
products

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
0,Official Redmi A3 | Layar Muluz 90 Hz berukura...,4.936567,1072.0,Xiaomi,Kota Depok,1199000.0
1,Infinix Note 40 8/256GB - Up to 16GB Extended ...,4.938308,1702.0,Infinix,Kota Depok,2729000.0
2,Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Ex...,4.931507,73.0,Infinix,Kota Depok,3579000.0
3,[PROMO] VIVO Y15S RAM 3GB / ROM 64GB - FULLSET...,5.000000,23.0,Mentari Cell,Kab. Madiun,999000.0
5,Xiaomi Redmi A3,4.987179,78.0,jikalaku,Kota Jakarta Pusat,1171000.0
...,...,...,...,...,...,...
3932,"samsung A05 Ram 6+128 GB kamera 50 mp, helio G...",5.000000,26.0,JIU_PHONCELL,Kab. Bogor,1385000.0
3933,TECNO Spark 10 - Ram 16GB (8+8GB) / 128GB - N...,4.875000,8.0,NISFU GADGET,Kab. Sidoarjo,1399000.0
3938,Infinix Hot 30i 8/128GB – Up to 16GB Extended ...,5.000000,2.0,ASK CELL,Kota Cimahi,1499000.0
3941,Infinix Note 30 Pro 8/256GB NEW Langsung Dikir...,3.000000,2.0,Zabeela Store,Kota Bandung,2799000.0


Dapat, dilihat, baris yang memiliki missing value berhasil dihapus.

In [97]:
products.isnull().sum()

,0
Name,0
Rating Score,0
Total Review,0
Seller Name,0
Location,0
Price (IDR),0


Setelah diperiksa, nilai missing values sudah tidak ada

#### TF-IDF Vectorizer

Teknik TF-IDF digunakan pada sistem rekomendasi untuk menemukan representasi fitur penting dari setiap kategori produk.

In [98]:
products

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
0,Official Redmi A3 | Layar Muluz 90 Hz berukura...,4.936567,1072.0,Xiaomi,Kota Depok,1199000.0
1,Infinix Note 40 8/256GB - Up to 16GB Extended ...,4.938308,1702.0,Infinix,Kota Depok,2729000.0
2,Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Ex...,4.931507,73.0,Infinix,Kota Depok,3579000.0
3,[PROMO] VIVO Y15S RAM 3GB / ROM 64GB - FULLSET...,5.000000,23.0,Mentari Cell,Kab. Madiun,999000.0
5,Xiaomi Redmi A3,4.987179,78.0,jikalaku,Kota Jakarta Pusat,1171000.0
...,...,...,...,...,...,...
3932,"samsung A05 Ram 6+128 GB kamera 50 mp, helio G...",5.000000,26.0,JIU_PHONCELL,Kab. Bogor,1385000.0
3933,TECNO Spark 10 - Ram 16GB (8+8GB) / 128GB - N...,4.875000,8.0,NISFU GADGET,Kab. Sidoarjo,1399000.0
3938,Infinix Hot 30i 8/128GB – Up to 16GB Extended ...,5.000000,2.0,ASK CELL,Kota Cimahi,1499000.0
3941,Infinix Note 30 Pro 8/256GB NEW Langsung Dikir...,3.000000,2.0,Zabeela Store,Kota Bandung,2799000.0


In [99]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
tf.fit(products['Name'])
tf.get_feature_names_out()

array(['000', '000mah', '007', ..., 'zero', 'zoom', 'zte'], dtype=object)

In [100]:
tfidf_matrix = tf.fit_transform(products['Name'])
tfidf_matrix.shape

(1720, 1176)

Matriks berukuran (1720, 1176). Nilai 1720 merupakan ukuran data dan 1176 merupakan matrik kategori produk atau banyaknya tipe dari kategori produk

In [101]:
tfidf_matrix.todense()

matrix([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]])

Selanjutnya, matriks tf-idf untuk nama produk dan kata kunci produk.

In [102]:
pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=products['Name'].values
).sample(50, axis=1).sample(50, axis=0)

,nightography,seluler,bnib,crystalres,sidik,quick,106,5f,6nm,6100,...,6gb_128gb,m6a,2mp,mension,waterdrop,2z,graansi,y85,os,smarter
XIAOMI REDMI 12C - RAM 4/64 GB - Chipset\tMediatek MT6769Z Helio G85 (12nm),0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Poco X6 Pro 5G 12/512 Garansi Resmi,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hp infinix Hot 40i NFC 16+256 GB unisog T606 (imei terdaftar) GARANSI RESMI 1 TAHUN,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
samsung A15 NFC Ram 8+128 GB layar Amoled fast charging 30 menit full GRATIS TRAVEL ADAPTOR garansi resmi 1 tahun ( pengganti A14),0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Oppo Reno 11 5G NFC 8 +256 GB Super Amoled & dimensity 8200 kamera 50 mp garansi 1 tahun resmi (Pengganti Reno 8),0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Samsung Galaxy M54 5G 8/256GB l Camera 108 MP l 6000mAh 25W Super Fast Charging l 6,7 inch super AMOLED+ Display l",0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
XIAOMI NOTE 12 PRO/XIAOMI 12 4G/5G - 6/128 & 8/256 - FULLSET - SECOND,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Vivo Y18 4/64GB 4/128GB 6/128GB RAM 4GB+4GB Extended ROM 64GB 5000mAh Y03 Garansi Resmi New 2024,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Vivo Y17 Ram 8/256GB Bergaransi 1 Tahun,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Infinix GT 20 Pro 5G 8/256GB - Garansi Resmi 1 Tahun,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Output sampel matriks tf-idf di atas menunjukkan 'REALME C33 4/128GB vs 4/64GB vs 3/32GB | OS Android 12, Realme UI S | Chipset Unisoc Tiger T612 (12 nm)' mengandung kata kunci '12'. Hal ini terlihat dari nilai matriks 0.277793 pada kategori kata kunci '12'. Begitu juga dengan 'Redmi 12 8/128/256GB MediaTek Helio G88 Garansi Resmi', termasuk dalam kategori kata kunci yang sama, dengan nilai 0.324161.

## Model Development dengan Content Based Filtering berdasarkan Kategori Produk

Teknik content based filtering akan merekomendasikan item yang mirip dengan item yang disukai pengguna di masa lalu. Pada tahap ini, ditemukan representasi fitur penting dari setiap kategori produk dengan tfidf vectorizer dan menghitung tingkat kesamaan dengan cosine similarity.

#### Cosine Similarity

Sekarang, dihitung derajat kesamaan (similarity degree) antar nama produk dengan teknik cosine similarity.

In [103]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.10037698, 0.        ,
        0.        ],
       [0.        , 1.        , 0.47791271, ..., 0.29828201, 0.0992759 ,
        0.        ],
       [0.        , 0.47791271, 1.        , ..., 0.2463192 , 0.1239093 ,
        0.        ],
       ...,
       [0.10037698, 0.29828201, 0.2463192 , ..., 1.        , 0.03714804,
        0.03280736],
       [0.        , 0.0992759 , 0.1239093 , ..., 0.03714804, 1.        ,
        0.04539743],
       [0.        , 0.        , 0.        , ..., 0.03280736, 0.04539743,
        1.        ]])

Dihitung cosine similarity dataframe tfidf_matrix yang  diperoleh pada tahapan sebelumnya. Dengan satu baris kode untuk memanggil fungsi cosine similarity dari library sklearn, hasil similarity tiap nama produk sudah didapat, hasilnya berupa matriks kesamaan dalam bentuk array.

In [104]:
cosine_sim_df = pd.DataFrame(cosine_sim, index=products['Name'], columns=products['Name'])
print('Shape:', cosine_sim_df.shape)

cosine_sim_df.sample(20, axis=1).sample(20, axis=0)

Shape: (1720, 1720)


Name,VIVO Y27s NFC Ram 16GB (8GB+8GB) / Rom 256GB - Snapdragon 680,FREE ADAPTER 25watt Original Samsung Galaxy A05 4/128GB Garansi Resmi,HP SAMSUNG A15 5G 8/256 NFC GARANSI RESMI SAMSUNG (GARANSI SOFTWARE GRATIS),Samsung Galaxy A15 8/128GB Garansi Resmi SEIN,REALME C51 4/128 vs 4/64 GB NFC - 33W CHAMPION CHARGE - 50MP AI CAMERA - BISA COD,SMARTPHONE A3S RAM 6/128GB FULLSET GARANSI 1 TAHUN,"REALME C33 4/128GB vs 4/64GB vs 3/32GB | OS Android 12, Realme UI S | Chipset Unisoc Tiger T612 (12 nm)",Xiaomi Poco X6 Pro 5G NFC - 12GB 512GB (12/512) AMOLED Garansi Resmi,itel P55 5G 6/128GB - Dual Camera 50MP - Fast Charge 18W Garansi resmi,Vivo Y28 8/256GB 8/128GB 6/128GB NFC 6000mAh Baterai 44W FastCharge Garansi Resmi Terbaru 2024 hp,Xiaomi Note 13 Pro Plus 5G 12/512GB - Garansi Resmi 1 Tahun,Realme C67 8/128GB NFC android,itel P55 5G RAM 6/128GB Garansi Resmi itel Indonesia,Itel P55 RAM 8/128 GB garansi resmi,"HP INFINIX SMART 8 RAM 8GB (4GB+4GB)/128GB TERBARU, 13MP CAMERA, 5000 MAH GARANSI RESMI 1TAHUN ( imei terdaftar )",Tecno Spark 20 NFC 8/128 & 8/256 GB Garansi Resmi,"oppo A,79 5G NFC ram 8+256 GB (imei terdaftar) garansi resmi 1 tahun oppo",Infinix Smart 8 3/64GB,INFINIX HOT 40 PRO 8/256 NEW GARANSI RESMI,realme C63 6GB+6GB*|128GB (45W Fast Charge | Air Gestures Control | TÜV Certification | 7.74mm Ultra Slim | NFC)
Name,,,,,,,,,,,,,,,,,,,,
HP Reno 2Z Ram 8/256 GB handphone smartphone HP Android jaringan 4G garansi 1bulan,0.015820,0.010297,0.130195,0.015728,0.020546,0.112752,0.031015,0.009956,0.010522,0.072956,0.012671,0.086289,0.031177,0.087391,0.082091,0.094727,0.090301,0.000000,0.066989,0.000000
Samsung a14 second rasa baru,0.000000,0.061600,0.123562,0.094090,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Redmi 13C,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
VIVO Y17 RAM 8 / 256 GB Original New Smarphone 6.35 in Handphone Murah,0.053741,0.067260,0.034661,0.000000,0.021107,0.024820,0.000000,0.000000,0.000000,0.031850,0.000000,0.000000,0.019402,0.072094,0.013120,0.083019,0.081650,0.000000,0.126662,0.000000
ITEL RS4 12/256 NFC NEW GARANSI RESMI,0.070545,0.037020,0.183638,0.056546,0.055417,0.027030,0.093665,0.184168,0.162860,0.091385,0.147582,0.110879,0.336285,0.266430,0.029883,0.223528,0.173845,0.000000,0.268396,0.040425
Vivo V29 5G 16+256 GB layar super Amoled snapdragon 778 G garansi resmi 1 tahun,0.124710,0.023612,0.101775,0.036067,0.022529,0.083977,0.000000,0.126111,0.054468,0.054055,0.119376,0.000000,0.063625,0.087422,0.019060,0.120520,0.170871,0.000000,0.092241,0.000000
REDMI 6A RAM 3/32GB GARANSI 1 TAHUN,0.025440,0.016557,0.033212,0.025291,0.000000,0.162005,0.099420,0.016010,0.016920,0.014066,0.099250,0.000000,0.050134,0.070213,0.033902,0.022377,0.111508,0.000000,0.025245,0.000000
Hp Vivo Y100 5G 8/256GB 80W FlashCharge 5000mAh Snapdragon 4 Gen 2 Vivo Terbaru 2024 Garansi Resmi,0.193896,0.023630,0.106419,0.036094,0.000000,0.017254,0.000000,0.051575,0.054509,0.314672,0.065643,0.000000,0.063673,0.039505,0.120515,0.031935,0.056064,0.000000,0.036029,0.000000
XIAOMI NOTE 12 PRO/XIAOMI 12 4G/5G - 6/128 & 8/256 - FULLSET - SECOND,0.000000,0.000000,0.077406,0.000000,0.028249,0.146050,0.138520,0.318901,0.035171,0.000000,0.489698,0.000000,0.041084,0.060122,0.000000,0.106393,0.081121,0.000000,0.148701,0.000000


Dengan cosine similarity, berhasil mengidentifikasi kesamaan antar produk lainnya. Shape (7636, 7636) merupakan ukuran matriks similarity dari data. Namun, dalam output, hanya menampilkan sebanyak 20 sampel data saja.


Contoh: angka 1.0 pada Skin Buddy Dot Burst Face Wash dan Bio Renew Deep Cleanser menunjukkan dua produk ini memiliki kesamaan. Begitu juga, dengan Oh! So Bright Serum dengan Skin'o'tic Serum, yang juga mendapat nilai 1.0

#### Mendapatkan Rekomendasi

Sistem rekomendasi akan memberikan produk yang memiliki similarity terhadap produk yang diinput oleh pengguna berdasarkan kesamaan kategori dari produk-produk rekomendasi, hasil similarity tiap produk sudah didapat dari perhitungan sebelumnya.

In [105]:
def products_recommendations(nama_produk, similarity_data=cosine_sim_df, items=products[['Name', 'Location', 'Seller Name']], k=10):
    """
    Rekomendasi Produk Berdasarkan Kesamaan Nama Produk.
    """
    # Mengambil data dengan similarity terbesar dari index yang diinputkan
    index = similarity_data.loc[:,nama_produk].to_numpy().argpartition(
        range(-1, -k-1, -1))

    # Mengambil data dengan similarity terbesar dari index yang diinputkan
    closest = similarity_data.columns[index[-1:-(k+2):-1]]

    # Drop nama_produk agar nama produk yang dicari tidak muncul dalam daftar rekomendasi
    closest = closest.drop(nama_produk, errors='ignore')

    # Membuat dataframe dari data rekomendasi
    recommendations = pd.DataFrame(closest, columns=['Name']).merge(items, on='Name') # Changed 'product_name' to 'Name' for merging

    return recommendations.head(k)

Dibuat fungsi dengan nama products_recommendations, dengan nama_produk sebagai parameter pencarian, hasil kesamaan yang diambil dari cosine_sim_df, isi dari dataframe yang ingin ditampilkan dan k (jumlah rekomendasi yang diinginkan) sebanyak 10.

Lalu membuat index untuk mengambil urutan indeks produk yang paling mirip, dengan similarity_data, dengan fungsi argpartition untuk mengurutkan indeks array berdasarkan skor kemiripan dari yang tertinggi ke terendah (berdasarkan parameter range(-1, -k-1, -1)).

Daftar produk disimpan dalam closest, dengan mengambil skor kemiripan tertinggi, lalu menghapus nama produk itu sendiri dari daftar rekomendasi. terakhir, dibuat variabel recommendation, untuk membuat dataframe dari data closest, column, dan items untuk digabung menjadi satu, dan nilai k dikembalikan.

In [106]:
products['Name'].unique()

array(['Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH',
       'Infinix Note 40 8/256GB - Up to 16GB Extended RAM - Helio G99 - 6.78" FHD+ Amoled 120HZ- 45 Watt Fast Charge - NFC',
       'Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Extended RAM - Dimensity 7020 - 6.78” FHD+ 3D Curved Amoled - 108MP OIS - 45W Charger - NFC',
       ...,
       'Infinix Hot 30i 8/128GB – Up to 16GB Extended RAM – Helio G37 - 6.6” HD+ IPS – 50MP AI Camera - 5000 mAh',
       'Infinix Note 30 Pro 8/256GB NEW Langsung Dikirim, Garansi Resmi',
       'OPPO A58 6/128GB Garansi Resmi Indonesia'], dtype=object)

In [107]:
products[products.Name.eq('OPPO A58 6/128GB Garansi Resmi Indonesia')]

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
3947,OPPO A58 6/128GB Garansi Resmi Indonesia,5.0,2.0,Apollo Gadget Store,Kota Surabaya,2399000.0


Output rekomendasi diharapkan akan memberikan produk serupa "OPPO A58 6/128GB Garansi Resmi Indonesia", dengan kategori produk yang mirip

In [108]:
products_recommendations('OPPO A58 6/128GB Garansi Resmi Indonesia')

,Name,Location,Seller Name
0,OPPO A58 6/128GB Garansi Resmi,Kota Malang,Alibabastore.id
1,OPPO A58 NFC (8GB+128GB) - GARANSI RESMI,Kota Jakarta Barat,Duniagadgetku
2,OPPO A58 NFC - RAM 8+8GB / ROM 128GB - GARANSI...,Kab. Sidoarjo,NISFU GADGET
3,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,J3 SHOP
4,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87
5,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,J3 SHOP
6,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87
7,HP OPPO A58 RAM 6GB ROM 128GB,Kab. Bekasi,Pelangipelangi Shop
8,Oppo A18 4/128GB Garansi Resmi Oppo Indonesia,Kota Bandung,Zabeela Store
9,Oppo A38 6/128GB 4/128GB Garansi Resmi Oppo In...,Kota Bandung,Zabeela Store


Output pun menampilkan 10 daftar produk rekomendasi yang memiliki kemiripan dengan "OPPO A58 6/128GB Garansi Resmi Indonesia", dengan Nama Seller dan asal seller nya juga

Menguji dengan nama produk lain

In [109]:
products_recommendations('Infinix Note 40 8/256GB - Up to 16GB Extended RAM - Helio G99 - 6.78" FHD+ Amoled 120HZ- 45 Watt Fast Charge - NFC')

,Name,Location,Seller Name
0,Infinix Note 40 PRO 8/256GB - Up to 16GB Exten...,Kota Depok,Infinix
1,Infinix Note 40 8/256GB - Up to 16GB Extended ...,Kab. Cirebon,MEDIKOM_88
2,Infinix Hot 40 Pro 8/256GB - Up to 16GB Extend...,Kota Jakarta Utara,Invens Store
3,Infinix Hot 40 Pro [8GB/256GB] Extended RAM 16...,Kab. Bekasi,Penabur
4,Infinix Hot 30 8/128GB – Up to 16GB Extended R...,Kota Cimahi,ASK CELL
5,Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Ex...,Kota Depok,Infinix
6,INFINIX HOT 40 PRO - Ram 16GB / Rom 256GB – H...,Kab. Sidoarjo,NISFU GADGET
7,Infinix Hot 40 Pro 8/256 – Helio G99 - 120Hz -...,Kota Depok,Infinix
8,Infinix Hot 40 Pro 12/256 – Helio G99 - 120Hz ...,Kota Depok,Infinix
9,"Hp infinix Hot 40 pro { NFC }RAM 8+8/256GB ,He...",Kab. Bogor,Riawan_shop88


In [110]:
products_recommendations('Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH')

,Name,Location,Seller Name
0,Xiaomi Redmi A3 4/128GB Garansi Resmi Layar Mu...,Kab. Tegal,Adiwerna DevStore
1,"Xiaomi Redmi A3 | Layar Muluz 90 Hz 6.71"" kapa...",Kota Surabaya,UFO ELEKTRONIKA DAN FURNITURE
2,Xiaomi Redmi A3,Kota Jakarta Pusat,jikalaku
3,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,hacom store surabaya
4,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official
5,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL
6,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,hacom store surabaya
7,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official
8,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL
9,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,hacom store surabaya


## Model Development dengan Content Based Filtering berdasarkan Rating Produk

Teknik content based filtering akan merekomendasikan item yang mirip dengan item yang disukai pengguna di masa lalu. Pada tahap ini, ditemukan representasi fitur penting dari setiap kategori produk dengan tfidf vectorizer dan menghitung tingkat kesamaan berdasarkan rating dengan cosine similarity.

### Data Preparation

##### TF-IDF Vectorizer

Teknik TF-IDF digunakan pada sistem rekomendasi untuk menemukan representasi fitur penting dari setiap kategori produk.

In [111]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
tf.fit(products['Name'])
tf.get_feature_names_out()

array(['000', '000mah', '007', ..., 'zero', 'zoom', 'zte'], dtype=object)

In [112]:
tfidf_matrix = tf.fit_transform(products['Name'])
tfidf_matrix.shape

(1720, 1176)

### Pemodelan

##### Cosine Similarity

Sekarang, dihitung derajat kesamaan (similarity degree) antar rating dari kategori yang sama pada produk dengan teknik cosine similarity.

In [113]:
total_rating = products[['Rating Score']].values
combined_features = np.hstack([total_rating, tfidf_matrix.toarray()])
cosine_sim_rating = cosine_similarity(combined_features)

Pada tahapan ini, dihitung cosine similarity dataframe tfidf_matrix yang diperoleh sebelumnya, lalu menyimpan nilai rating dalam variabel total_rating, dan menggabungkan dalam bentuk array dari hasil tfidf_matrix dengan nilai rating yang sudah disimpan

In [114]:
cosine_sim_df_2 = pd.DataFrame(cosine_sim_rating, index=products['Name'], columns=products['Name'])
print('Shape:', cosine_sim_df_2.shape)

cosine_sim_df_2.sample(20, axis=1).sample(20, axis=0)

Shape: (1720, 1720)


Name,"Xiaomi Redmi A3 | Layar Muluz 90 Hz 6.71"" kapasitas baterai 5000 mAH",oppo A38 6/128 GB big batrai 5000 mah garansi resmi 1 tahun,[LIKE NEW] INFINIX ZERO 30 8/256 4G - 3D CURVED AMOLED SCREEN - 50MP FRONT VLOG CAMERA,Xiaomi Redmi Note 9 Pro Certified Renewed Unit Only Grade A,Apple iPhone 12,Samsung A34 5G 8/128GB | 8/256GB | Garansi Resmi 1 Tahun,【HP BARU】【BISA COD】VIVO V23 5G RAM 8/256 GB ORI Android 64MP Camera Garansi 12 Bulan,OPPO A15S RAM 8/256 GB,Samsung A04S 4/64,Realme C15 Ram 4/64GB Fullset,SMARTPHONE 12 RAM 8/256GB GARANSI 1 THN BISA COD,"HP SAMSUNG A05 RAM 6GB+6GB /128GB HELIO G85 CAM 50MP, BATRAI 5000MAH, GARANSI RESMI 1TAHUN",REALME 10 4G 8/256 GB | REALME10 8/128 GB GARANSI RESMI REALME,"[FREE MINI BAG] TECNO PHANTOM V Flip 5G - 8+8GB*+256GB, 64MP Ultra-clear Cam, 120Hz Screen, 4000mAh+45W Flash Charging, NFC, The Planet Cover Screen, 32MP Dual-flash Selfie, Android 13, Garansi 12+1 Bulan",REALME C53 NFC 8/256 vs 6/128 - 50MP AI CAMERA - 33W CHAMPION CHARGE - BISA COD,Vivo Y27 6+128GB | 50MP | Mediatek MT6769 Helio G85 (12nm) | 5000 mAh [BISA COD],Oppo A18 | Oppo A38 4/128GB 6.56inci Helio G85 Prosesor 5000mAh Oppo A58 Terbaru 2023 Garansi Resmi,SAMSUNG GALAXY A05S 6/128 & NEW GARANSI RESMI,SAMSUNG GALAXY A05S RAM 6 ROM 128 - SNAPDRAGON 680 - FHD+,VIVO Y20s Ram 8+256GB COD Garansi 12 Bulan
Name,,,,,,,,,,,,,,,,,,,,
Infinix Hot 11 Ram 4GB / 64GB Garansi Resmi,0.961538,0.963099,0.963692,0.877058,0.961194,0.963772,0.951367,0.963383,0.961538,0.968056,0.963029,0.963704,0.960402,0.961828,0.956420,0.960777,0.962081,0.962683,0.961437,0.963889
PROMO HP INFINIX HOT 20i RAM 4/64 BONUS TEMPER GLAS MULUS - RAM 4/64 GB,0.955927,0.957018,0.957167,0.871940,0.955585,0.955927,0.948062,0.959805,0.963744,0.957669,0.956692,0.958244,0.955439,0.955927,0.950838,0.955170,0.955367,0.954517,0.956033,0.957566
OPPO A18 RAM 4GB/128GB Garansi Resmi - Grandivo,0.961538,0.965570,0.961538,0.877058,0.961194,0.965807,0.951252,0.967399,0.961538,0.962965,0.962891,0.964789,0.960312,0.961811,0.956420,0.961961,0.973589,0.962537,0.961362,0.963755
Realme c33 ram 4/64 masih mulus no minus graansi resmi Indonesia,0.961538,0.961986,0.961538,0.877058,0.961194,0.962180,0.950020,0.962553,0.965275,0.965124,0.961419,0.962402,0.963516,0.961538,0.958287,0.960777,0.961292,0.960856,0.960845,0.962322
Hp Vivo Y03 4/128GB 5000mAh + 3 Tahun Proteksi Baterai Vivo Hp Murah 100% Original Garansi Resmi,0.963164,0.963143,0.960437,0.876053,0.960092,0.965707,0.955738,0.960437,0.960437,0.960437,0.960092,0.966406,0.958682,0.960613,0.955324,0.962964,0.962800,0.960579,0.959020,0.964998
HP BARU SAMSUNG GALAXY A35 5G | A55 5G 8/128GB - 12/256GB IP67 Water Resistance 100% ORI GARANSI RESMI,0.961538,0.962290,0.961538,0.877058,0.963833,0.969568,0.962126,0.961538,0.963627,0.961538,0.964401,0.964830,0.959579,0.963422,0.956420,0.961382,0.962054,0.965898,0.963298,0.965225
Infinix Hot 40 pro NFC 12GB/256GB Garansi Resmi - Grandivo,0.959969,0.961263,0.961755,0.880247,0.959625,0.964098,0.948457,0.959969,0.959969,0.959969,0.961865,0.961004,0.958566,0.961781,0.957066,0.959209,0.960323,0.960678,0.958553,0.962713
"HP MAXTRON P12i SUPER -- HP CANDYBAR 2,4"" - BIG SPEAKER - HP OUTDOOR - HP GUNUNG - HP POWERBANK - BLUETOOTH SPEAKER - HP MAXTRON P12 - Dual Sim - Powerbank / Fm Radio / hp murah / hp keren / Laz Promo Murah Nampol",0.958433,0.960730,0.958433,0.874226,0.958090,0.958433,0.951726,0.958433,0.958433,0.958433,0.957512,0.962513,0.955718,0.958883,0.953331,0.957674,0.957871,0.957019,0.957019,0.958433
[Hot] itel S23+ 256GB+16GB(8+8GB) Unisoc Tiger T616 5000 mAh 18W 6.78 inches- AMOLED CURVED - 32MP AI SELFIE - NFC,0.964496,0.966203,0.965066,0.876325,0.960390,0.961894,0.948552,0.960734,0.960734,0.960734,0.960844,0.960734,0.958012,0.965634,0.959186,0.964264,0.960171,0.959317,0.959317,0.961741


Dengan cosine similarity, berhasil mengidentifikasi kesamaan antar produk lainnya. Shape (1720, 1720) merupakan ukuran matriks similarity dari data. Namun, dalam output, hanya menampilkan sebanyak 20 sampel data saja.


Contoh: angka 	0.961239 pada "Apple iPhone 14" dan "Samsung Galaxy A15 5G 8/256GB" menunjukkan dua produk ini memiliki kesamaan kategori dan rating yang serupa

##### Mendapatkan Rekomendasi

Sistem rekomendasi akan memberikan produk yang memiliki similarity terhadap produk yang diinput oleh pengguna berdasarkan kesamaan kategori dan rating yang paling tinggi dari produk-produk rekomendasi, hasil similarity tiap produk sudah didapat dari perhitungan sebelumnya.

In [115]:
def products_recommendations_by_rating(nama_produk, similarity_data=cosine_sim_df_2, items=products[['Name', 'Location', 'Seller Name',
                                                                                                     'Rating Score']], k=10):
    index = similarity_data.loc[:,nama_produk].to_numpy().argpartition(
        range(-1, -k-1, -1))

    closest = similarity_data.columns[index[-1:-(k+2):-1]]
    closest = closest.drop(nama_produk, errors='ignore')
    recommendations = pd.DataFrame(closest, columns=['Name']).merge(items, on='Name')
    recommendations = recommendations.sort_values(by='Rating Score', ascending=False)

    return recommendations.head(k)

Dibuat fungsi dengan nama products_recommendations, dengan nama_produk sebagai parameter pencarian, hasil kesamaan yang diambil dari cosine_sim_df, isi dari dataframe yang ingin ditampilkan dan k (jumlah rekomendasi yang diinginkan) sebanyak 10.

Lalu membuat index untuk mengambil urutan indek produk yang paling mirip, dengan similarity_data, dengan fungsi argpartition untuk mengurutkan indeks array berdasarkan skor kemiripan dari yang tertinggi ke terendah (berdasarkan parameter range(-1, -k-1, -1)).

Daftar produk disimpan dalam closest, dengan mengambil skor kemiripan tertinggi, lalu menghapus nama produk itu sendiri dari daftar rekomendasi. terakhir, dibuat variabel recommendation, untuk membuat dataframe dari data closest, column, dan items untuk digabung menjadi satu, diberikan tambahan untuk mengurutkan produk rekomendasi dari rating yang paling tinggi dan nilai k dikembalikan.

In [116]:
products['Name'].unique()

array(['Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH',
       'Infinix Note 40 8/256GB - Up to 16GB Extended RAM - Helio G99 - 6.78" FHD+ Amoled 120HZ- 45 Watt Fast Charge - NFC',
       'Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Extended RAM - Dimensity 7020 - 6.78” FHD+ 3D Curved Amoled - 108MP OIS - 45W Charger - NFC',
       ...,
       'Infinix Hot 30i 8/128GB – Up to 16GB Extended RAM – Helio G37 - 6.6” HD+ IPS – 50MP AI Camera - 5000 mAh',
       'Infinix Note 30 Pro 8/256GB NEW Langsung Dikirim, Garansi Resmi',
       'OPPO A58 6/128GB Garansi Resmi Indonesia'], dtype=object)

In [117]:
products[products.Name.eq('Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH')]

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
0,Official Redmi A3 | Layar Muluz 90 Hz berukura...,4.936567,1072.0,Xiaomi,Kota Depok,1199000.0


Output rekomendasi diharapkan akan memberikan produk serupa "Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH", dengan kategori yang mirip.

In [118]:
products_recommendations_by_rating('Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH')

,Name,Location,Seller Name,Rating Score
0,Xiaomi Redmi A3 4/128GB Garansi Resmi Layar Mu...,Kab. Tegal,Adiwerna DevStore,5.000000
1,"Xiaomi Redmi A3 | Layar Muluz 90 Hz 6.71"" kapa...",Kota Surabaya,UFO ELEKTRONIKA DAN FURNITURE,5.000000
5,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL,5.000000
4,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official,5.000000
8,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL,5.000000
7,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official,5.000000
13,Xiaomi REDMI A3 4/128 GB - Garansi Resmi,Kota Jakarta Timur,Royal Makmur Handphone & Tablet,5.000000
11,Xiaomi Redmi A3 Ram 4/128Gb Garansi Resmi,Kota Kediri,Cv.Garden Cell,5.000000
9,xiaomi redmi A3 4/128 Garansi resmi,Kota Jakarta Pusat,Galeri-Hp 040595,5.000000
2,Xiaomi Redmi A3,Kota Jakarta Pusat,jikalaku,4.987179


Output pun menampilkan 10 daftar produk rekomendasi yang memiliki kemiripan dengan "Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH", dengan nama seller, asal seller, dan nilai rating dari yang paling tinggi ke terendah

In [119]:
products_recommendations_by_rating('OPPO A58 6/128GB Garansi Resmi Indonesia')

,Name,Location,Seller Name,Rating Score
0,OPPO A58 6/128GB Garansi Resmi,Kota Malang,Alibabastore.id,5.0
1,OPPO A58 NFC (8GB+128GB) - GARANSI RESMI,Kota Jakarta Barat,Duniagadgetku,5.0
2,OPPO A58 NFC - RAM 8+8GB / ROM 128GB - GARANSI...,Kab. Sidoarjo,NISFU GADGET,5.0
4,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87,5.0
10,OPPO A18 4/128GB Garansi Resmi Indonesia,Kota Surabaya,Apollo Gadget Store,5.0
6,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87,5.0
7,HP OPPO A58 RAM 6GB ROM 128GB,Kab. Bekasi,Pelangipelangi Shop,5.0
8,Oppo A18 4/128GB Garansi Resmi Oppo Indonesia,Kota Bandung,Zabeela Store,5.0
11,OPPO A38 RAM 4/128GB & 6/128GB (EXTENDED RAM) ...,Kab. Tangerang,Rajalaku Store.Id,5.0
9,Oppo A38 6/128GB 4/128GB Garansi Resmi Oppo In...,Kota Bandung,Zabeela Store,5.0


## Model Development dengan Content Based Filtering berdasarkan Price (IDR)

Teknik content based filtering akan merekomendasikan item berdasarkan harga. Pada tahap ini, ditemukan representasi fitur penting dari setiap kategori produk dengan tfidf vectorizer dan menghitung tingkat kesamaan berdasarkan total rekomendasi dengan cosine similarity.

### Data Preparation

##### TF-IDF Vectorizer

Teknik TF-IDF digunakan pada sistem rekomendasi untuk menemukan representasi fitur penting dari setiap kategori produk.

In [120]:
products

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
0,Official Redmi A3 | Layar Muluz 90 Hz berukura...,4.936567,1072.0,Xiaomi,Kota Depok,1199000.0
1,Infinix Note 40 8/256GB - Up to 16GB Extended ...,4.938308,1702.0,Infinix,Kota Depok,2729000.0
2,Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Ex...,4.931507,73.0,Infinix,Kota Depok,3579000.0
3,[PROMO] VIVO Y15S RAM 3GB / ROM 64GB - FULLSET...,5.000000,23.0,Mentari Cell,Kab. Madiun,999000.0
5,Xiaomi Redmi A3,4.987179,78.0,jikalaku,Kota Jakarta Pusat,1171000.0
...,...,...,...,...,...,...
3932,"samsung A05 Ram 6+128 GB kamera 50 mp, helio G...",5.000000,26.0,JIU_PHONCELL,Kab. Bogor,1385000.0
3933,TECNO Spark 10 - Ram 16GB (8+8GB) / 128GB - N...,4.875000,8.0,NISFU GADGET,Kab. Sidoarjo,1399000.0
3938,Infinix Hot 30i 8/128GB – Up to 16GB Extended ...,5.000000,2.0,ASK CELL,Kota Cimahi,1499000.0
3941,Infinix Note 30 Pro 8/256GB NEW Langsung Dikir...,3.000000,2.0,Zabeela Store,Kota Bandung,2799000.0


In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
tf.fit(products['Name'])
tf.get_feature_names_out()

array(['000', '000mah', '007', ..., 'zero', 'zoom', 'zte'], dtype=object)

In [122]:
tfidf_matrix = tf.fit_transform(products['Name'])
tfidf_matrix.shape

(1720, 1176)

### Pemodelan

##### Cosine Similarity

Sekarang, dihitung derajat kesamaan (similarity degree) antar total rekomendasi dari kategori yang sama pada produk dengan teknik cosine similarity.

In [123]:
total_recommended_count = products[['Price (IDR)']].values
combined_features = np.hstack([total_recommended_count, tfidf_matrix.toarray()])
cosine_sim_recom = cosine_similarity(combined_features)

Pada tahapan ini, dihitung cosine similarity dataframe tfidf_matrix yang diperoleh sebelumnya, lalu menyimpan total rekomendasi dalam variabel total_recommended_count, dan menggabungkan dalam bentuk array dari hasil tfidf_matrix dengan total rekomendasi yang sudah disimpan

In [124]:
cosine_sim_df_3 = pd.DataFrame(cosine_sim_recom, index=products['Name'], columns=products['Name'])
print('Shape:', cosine_sim_df_3.shape)

cosine_sim_df_3.sample(20, axis=1).sample(20, axis=0)

Shape: (1720, 1720)


Name,Apple iPhone 15 Plus,【Beli 1 Gratis 6】hp murah GALAXY A04e 5G RAM 16GB+ROM 512GB 7.5Inci Handphone android galaxy A13 Garansi Resmi 1000 ribuan hp murah promo cuci gudang cod asli terbaru Indonesia siap Gratis ongkir 100% Original Handphone murah promo cuci gudang hp murah,XIAOMI REDMI NOTE 5 RAM 4/64GB & 6/64GB & 6/128GB Smartphone 4G LTE Garansi Distributor HP,OPPO A18 RAM 4/128GB l BARU l SEGEL l GARANSI RESMI 1 TAHUN,"[New POVA Series] TECNO POVA 5 Ram 8+8GB Internal 256GB, Mediatek Helio G99, 6000mAh +45W Flash Charge","Vivo Smartphone V27 5G 8/256GB 6,78 Inch Garansi Resmi","INFINIX HOT 40 PRO - Ram 16GB / Rom 256GB – Helio G99 - 120Hz - 6.78"" FHD+ Hypervision Gaming Pro Display - 5000mAh",Redmi Note 13 5G,Vivo Y27s NFC 8/256 GB 8/128 GB Garansi Resmi,Infinix Note 40 8/256GB - Up to 16GB Extended RAM - Helio G99 - 6.78” FHD+ Amoled 120Hz - Camera 108MP - 5000 mAh - NFC,ITEL A70 4/256 & 4/128 & 4/64 NEW GARANSI RESMI,SMARTPHONE F1S RAM 4/64GB FULLSET GARANSI 1 TAHUN,Infinix Hot 30 RAM 8GB/256GB Garansi Resmi - Grandivo,"Infinix Smart 8 [ 4GB/128GB] Unisoc T606 - 6.6"" IPS LCD - 5000mAh Garansi Resmi 1 Tahun",MAXTRON P12 BOMBA - HP OUTDOOR - LAYAR 2.4 INCH - BIG BATTERY - CAMERA- MP3 - RADIO - BISA POWERBANK - GARANSI RESMI,Smartphone 11 64GB Garansi Resmi,Xiaomi Redmi Note 12 Pro 4G - Garansi resmi,INFINIX HOT 40 PRO 8/256 NEW GARANSI RESMI,Vivo Y28 8/128GB RAM 8GB+8GB Extended ROM 256GB 44W FlashCharge Vivo Terbaru 2024 Garansi Resmi,INFINIX Hot 30i 8/128 Garansi Resmi
Name,,,,,,,,,,,,,,,,,,,,
"【Produk Baru HP】Ponsel Pintar Baru Phone i14 Pro Max 7,5 inch HD Ponsel asli resmi Kamera Ponsel Android RAM 16GB + ROM 512GB Siswa Belajar Ponsel dual card dual standby gaming phone hp murah Baru Asli Smartphone",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Redmi 13,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
EVERCOSS U6 XTREAM 1 PLUS RAM 1GB INTERNAL 8GB GARANSI RESMI,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Infinix Hot 40 Pro 8/256GB & 12/256GB - Garansi Resmi 1 Tahun,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Evercoss Xtream 1 (S45) garansi resmi evercoss indonesia,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
SAMSUNG GALAXY S23 FE 5G 8/256GB - 8/128GB NEW!! GARANSI RESMI SEIN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Itel A70 3/128 4/64 4/128 & 8/256 GB Garansi Resmi,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
"Realme Note 50 4/64GB & 4/128GB, Unisoc Tiger T612 | Bergaransi",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
Oppo A16K 4/64 Garansi Resmi,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Dengan cosine similarity, berhasil mengidentifikasi kesamaan antar produk lainnya. Shape (1720, 1720) merupakan ukuran matriks similarity dari data. Namun, dalam output, hanya menampilkan sebanyak 20 sampel data saja.


Contoh: angka 	0.961239 pada "Apple iPhone 14" dan "Samsung Galaxy A15 5G 8/256GB" menunjukkan dua produk ini memiliki kesamaan kategori dan rating yang serupa

##### Mendapatkan Rekomendasi

Sistem rekomendasi akan memberikan produk yang memiliki similarity terhadap produk yang diinput oleh pengguna berdasarkan kesamaan kategori dan rating yang paling tinggi dari produk-produk rekomendasi, hasil similarity tiap produk sudah didapat dari perhitungan sebelumnya.

In [125]:
def products_recommendations_by_count_recom(nama_produk, similarity_data=cosine_sim_df_2, items=products[['Name', 'Location', 'Seller Name',
                                                                                                     'Price (IDR)']], k=10):
    index = similarity_data.loc[:,nama_produk].to_numpy().argpartition(
        range(-1, -k-1, -1))

    closest = similarity_data.columns[index[-1:-(k+2):-1]]
    closest = closest.drop(nama_produk, errors='ignore')
    recommendations = pd.DataFrame(closest, columns=['Name']).merge(items, on='Name')
    recommendations = recommendations.sort_values(by='Price (IDR)', ascending=False)

    return recommendations.head(k)

Dibuat fungsi dengan nama products_recommendations, dengan nama_produk sebagai parameter pencarian, hasil kesamaan yang diambil dari cosine_sim_df, isi dari dataframe yang ingin ditampilkan dan k (jumlah rekomendasi yang diinginkan) sebanyak 10.

Lalu membuat index untuk mengambil urutan indek produk yang paling mirip, dengan similarity_data, dengan fungsi argpartition untuk mengurutkan indeks array berdasarkan skor kemiripan dari yang tertinggi ke terendah (berdasarkan parameter range(-1, -k-1, -1)).

Daftar produk disimpan dalam closest, dengan mengambil skor kemiripan tertinggi, lalu menghapus nama produk itu sendiri dari daftar rekomendasi. terakhir, dibuat variabel recommendation, untuk membuat dataframe dari data closest, column, dan items untuk digabung menjadi satu, diberikan tambahan untuk mengurutkan produk rekomendasi dari total rekomendasi yang paling banyak dan nilai k dikembalikan.

In [126]:
products['Name'].unique()

array(['Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH',
       'Infinix Note 40 8/256GB - Up to 16GB Extended RAM - Helio G99 - 6.78" FHD+ Amoled 120HZ- 45 Watt Fast Charge - NFC',
       'Infinix Note 40 Pro 5G 8/256GB - Up to 16GB Extended RAM - Dimensity 7020 - 6.78” FHD+ 3D Curved Amoled - 108MP OIS - 45W Charger - NFC',
       ...,
       'Infinix Hot 30i 8/128GB – Up to 16GB Extended RAM – Helio G37 - 6.6” HD+ IPS – 50MP AI Camera - 5000 mAh',
       'Infinix Note 30 Pro 8/256GB NEW Langsung Dikirim, Garansi Resmi',
       'OPPO A58 6/128GB Garansi Resmi Indonesia'], dtype=object)

In [127]:
products[products.Name.eq('OPPO A58 6/128GB Garansi Resmi Indonesia')]

,Name,Rating Score,Total Review,Seller Name,Location,Price (IDR)
3947,OPPO A58 6/128GB Garansi Resmi Indonesia,5.0,2.0,Apollo Gadget Store,Kota Surabaya,2399000.0


Output rekomendasi diharapkan akan memberikan produk serupa "OPPO A58 6/128GB Garansi Resmi Indonesia", dengan kategori yang mirip

In [128]:
products_recommendations_by_count_recom('OPPO A58 6/128GB Garansi Resmi Indonesia')

,Name,Location,Seller Name,Price (IDR)
3,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,J3 SHOP,2469000.0
5,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,J3 SHOP,2469000.0
1,OPPO A58 NFC (8GB+128GB) - GARANSI RESMI,Kota Jakarta Barat,Duniagadgetku,2449000.0
0,OPPO A58 6/128GB Garansi Resmi,Kota Malang,Alibabastore.id,2399000.0
7,HP OPPO A58 RAM 6GB ROM 128GB,Kab. Bekasi,Pelangipelangi Shop,2399000.0
4,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87,2125000.0
6,OPPO A58 6/128 & 8/128 NEW GARANSI RESMI,Kota Medan,KSTORE87,2125000.0
9,Oppo A38 6/128GB 4/128GB Garansi Resmi Oppo In...,Kota Bandung,Zabeela Store,1799000.0
11,OPPO A38 RAM 4/128GB & 6/128GB (EXTENDED RAM) ...,Kab. Tangerang,Rajalaku Store.Id,1675000.0
2,OPPO A58 NFC - RAM 8+8GB / ROM 128GB - GARANSI...,Kab. Sidoarjo,NISFU GADGET,1599000.0


Output pun menampilkan 10 daftar produk rekomendasi yang memiliki kemiripan dengan "OPPO A58 6/128GB Garansi Resmi Indonesia", dengan rentang harga, asal brandnya, dan total rekomendasi dari yang paling mahal ke yang murah.

In [129]:
products_recommendations_by_count_recom('Official Redmi A3 | Layar Muluz 90 Hz berukuran 6.71" kapasitas baterai 5000 mAH')

,Name,Location,Seller Name,Price (IDR)
10,Official Xiaomi Redmi 10C (4/128GB) Snapdragon...,Kota Depok,Xiaomi,1499000.0
0,Xiaomi Redmi A3 4/128GB Garansi Resmi Layar Mu...,Kab. Tegal,Adiwerna DevStore,1200000.0
5,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL,1199000.0
4,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official,1199000.0
6,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,hacom store surabaya,1199000.0
3,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,hacom store surabaya,1199000.0
8,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Surabaya,OBONG CELL,1199000.0
7,Xiaomi Redmi A3 4/128GB Garansi Resmi,Kota Jakarta Pusat,Okeshop Official,1199000.0
9,xiaomi redmi A3 4/128 Garansi resmi,Kota Jakarta Pusat,Galeri-Hp 040595,1198000.0
13,Xiaomi REDMI A3 4/128 GB - Garansi Resmi,Kota Jakarta Timur,Royal Makmur Handphone & Tablet,1188000.0
